# Experiment-5: Subword Tokenization and POS Tagging

## 5.1. Implement subword-level tokenization using a BPE-based tokenizer.
(Implement two separate codes: a. Using pretrained model and b. Without using pretrained model)

### a. Using a pretrained BPE tokenizer (GPT-2)

In [1]:
from transformers import GPT2Tokenizer
import pandas as pd

In [2]:
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

sentence = (
    "naturalization, internationalization, computationally, unbelievable, "
    "preprocessing, classification, communication, renewable, hydrokinetic, "
    "multilingual, understanding, and modernization"
)

In [3]:
tokens = gpt2_tokenizer.tokenize(sentence)
token_ids = gpt2_tokenizer.convert_tokens_to_ids(tokens)

df_bpe_pretrained = pd.DataFrame({"Token": tokens, "Token ID": token_ids})
df_bpe_pretrained

,Token,Token ID
0,natural,11802
1,ization,1634
2,",",11
3,Ġinternational,3230
4,ization,1634
5,",",11
6,Ġcomput,2653
7,ationally,15208
8,",",11
9,Ġunbelievable,24479


### b. Without using a pretrained model (BPE trained from scratch on the corpus)

In [4]:
import re
from collections import Counter, defaultdict

In [5]:
file_path = "input_sub_word_data.txt"

with open(file_path, "r", encoding="utf-8") as f:
    corpus_text = f.read()

words = re.findall(r"[a-z]+", corpus_text.lower())
word_freqs = Counter(" ".join(list(word)) + " </w>" for word in words)

In [6]:
def get_pair_freqs(word_freqs):
    pairs = defaultdict(int)
    for word, freq in word_freqs.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs


def merge_pair(pair, word_freqs):
    bigram = re.escape(" ".join(pair))
    pattern = re.compile(r"(?<!\S)" + bigram + r"(?!\S)")
    return {pattern.sub("".join(pair), word): freq for word, freq in word_freqs.items()}

In [7]:
num_merges = 200
merges = []

for _ in range(num_merges):
    pair_freqs = get_pair_freqs(word_freqs)
    if not pair_freqs:
        break
    best_pair = max(pair_freqs, key=pair_freqs.get)
    word_freqs = merge_pair(best_pair, word_freqs)
    merges.append(best_pair)

vocab = {}
for word in word_freqs:
    for symbol in word.split():
        vocab.setdefault(symbol, len(vocab))

len(merges), len(vocab)

(200, 205)

In [8]:
def bpe_tokenize_word(word, merges):
    symbols = list(word.lower()) + ["</w>"]
    for pair in merges:
        i = 0
        new_symbols = []
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                new_symbols.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols

In [9]:
words_to_tokenize = re.findall(r"[a-zA-Z]+", sentence)

bpe_tokens, bpe_ids = [], []
for word in words_to_tokenize:
    for token in bpe_tokenize_word(word, merges):
        bpe_tokens.append(token)
        bpe_ids.append(vocab.get(token, -1))

df_bpe_scratch = pd.DataFrame({"Token": bpe_tokens, "Token ID": bpe_ids})
df_bpe_scratch

,Token,Token ID
0,nat,204
1,ur,154
2,al,73
3,ization</w>,186
4,in,15
...,...,...
64,and</w>,31
65,mod,165
66,er,34
67,n,97


## 5.2. Implement subword-level tokenization using a SentencePiece tokenizer.
(Implement two separate codes: a. Using pretrained model and b. Without using pretrained model)

### a. Using a pretrained SentencePiece tokenizer (ALBERT)

In [10]:
from transformers import AlbertTokenizer

sp_pretrained_tokenizer = AlbertTokenizer.from_pretrained("albert-base-v2")

In [11]:
sp_tokens = sp_pretrained_tokenizer.tokenize(sentence)
sp_ids = sp_pretrained_tokenizer.convert_tokens_to_ids(sp_tokens)

df_sp_pretrained = pd.DataFrame({"Token": sp_tokens, "Token ID": sp_ids})
df_sp_pretrained

,Token,Token ID
0,▁natural,1112
1,ization,1829
2,",",15
3,▁international,294
4,ization,1829
5,",",15
6,▁computational,16439
7,ly,102
8,",",15
9,▁un,367


### b. Without using a pretrained model (SentencePiece trained from scratch on the corpus)

In [12]:
import os
import sentencepiece as spm

with open(os.devnull, "w") as log_file:
    spm.SentencePieceTrainer.train(
        input=file_path,
        model_prefix="corpus_sentencepiece",
        vocab_size=400,
        model_type="unigram",
        logstream=log_file,
    )

I0000 00:00:1788231417.141566  978553 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: input_sub_word_data.txt
  input_format: 
  model_prefix: corpus_sentencepiece
  model_type: UNIGRAM
  vocab_size: 400
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_su

In [13]:
sp_scratch = spm.SentencePieceProcessor(model_file="corpus_sentencepiece.model")

scratch_tokens = sp_scratch.encode(sentence, out_type=str)
scratch_ids = sp_scratch.encode(sentence, out_type=int)

df_sp_scratch = pd.DataFrame({"Token": scratch_tokens, "Token ID": scratch_ids})
df_sp_scratch

,Token,Token ID
0,▁natural,59
1,ization,31
2,",",4
3,▁international,78
4,ization,31
5,",",4
6,▁computational,190
7,ly,96
8,",",4
9,▁un,43


## 5.3. Implement Part-of-Speech (POS) tagging on a given text and display the unique token, POS tag, and description of each POS tag.

"The young student is reading an interesting book in the library."

(Implement two separate codes: a. Using Spacy and b. NLTK)

### a. Using spaCy

In [14]:
import spacy

nlp = spacy.load("en_core_web_sm")

pos_sentence = "The young student is reading an interesting book in the library."
doc = nlp(pos_sentence)

In [15]:
seen_tokens = set()
spacy_rows = []
for token in doc:
    if token.is_space or token.text in seen_tokens:
        continue
    seen_tokens.add(token.text)
    spacy_rows.append((token.text, token.pos_, spacy.explain(token.pos_)))

df_pos_spacy = pd.DataFrame(spacy_rows, columns=["Token", "POS Tag", "Description"])
df_pos_spacy

,Token,POS Tag,Description
0,The,DET,determiner
1,young,ADJ,adjective
2,student,NOUN,noun
3,is,AUX,auxiliary
4,reading,VERB,verb
5,an,DET,determiner
6,interesting,ADJ,adjective
7,book,NOUN,noun
8,in,ADP,adposition
9,the,DET,determiner


### b. Using NLTK

In [16]:
import nltk

nltk_tokens = nltk.word_tokenize(pos_sentence)
nltk_tagged = nltk.pos_tag(nltk_tokens)
tag_descriptions = dict(nltk.data.load("help/tagsets/upenn_tagset.pickle"))

/opt/homebrew/lib/python3.11/site-packages/nltk/app/__init__.py:29: UserWarning: nltk.app package not loaded (please install Tkinter library).
  warnings.warn("nltk.app package not loaded (please install Tkinter library).")


In [17]:
seen_tokens = set()
nltk_rows = []
for word, tag in nltk_tagged:
    if word in seen_tokens:
        continue
    seen_tokens.add(word)
    description = tag_descriptions.get(tag, (tag,))[0]
    nltk_rows.append((word, tag, description))

df_pos_nltk = pd.DataFrame(nltk_rows, columns=["Token", "POS Tag", "Description"])
df_pos_nltk

,Token,POS Tag,Description
0,The,DT,determiner
1,young,JJ,"adjective or numeral, ordinal"
2,student,NN,"noun, common, singular or mass"
3,is,VBZ,"verb, present tense, 3rd person singular"
4,reading,VBG,"verb, present participle or gerund"
5,an,DT,determiner
6,interesting,JJ,"adjective or numeral, ordinal"
7,book,NN,"noun, common, singular or mass"
8,in,IN,"preposition or conjunction, subordinating"
9,the,DT,determiner


## 5.4. Implement Part-of-Speech (POS) tagging on a given text and display the unique token, POS tag, description of each token and frequency using Spacy.

In [18]:
corpus_doc = nlp(corpus_text)
corpus_tokens = [token for token in corpus_doc if not token.is_space]

token_freq = Counter(token.text for token in corpus_tokens)

In [19]:
first_occurrence = {}
for token in corpus_tokens:
    first_occurrence.setdefault(token.text, token)

pos_freq_rows = [
    (text, tok.pos_, spacy.explain(tok.pos_), token_freq[text])
    for text, tok in first_occurrence.items()
]

df_pos_freq = pd.DataFrame(
    pos_freq_rows, columns=["Token", "POS Tag", "Description", "Frequency"]
)
df_pos_freq = df_pos_freq.sort_values("Frequency", ascending=False).reset_index(drop=True)
df_pos_freq

,Token,POS Tag,Description,Frequency
0,",",PUNCT,punctuation,76
1,.,PUNCT,punctuation,65
2,the,DET,determiner,39
3,and,CCONJ,coordinating conjunction,27
4,can,AUX,auxiliary,23
...,...,...,...,...
390,translation,NOUN,noun,1
391,question,NOUN,noun,1
392,answering,NOUN,noun,1
393,summarization,NOUN,noun,1
